# Lab 0 — First contact

**~30 minutes.** Adapted from Week 1 of Ed Donner's *LLM Engineering* course, rebuilt on a
free stack: no credit card, no OpenAI billing.

By the end you will have:

1. a working environment and a call to a hosted model,
2. optionally the same call against a model running on your own laptop,
3. a feel for tokens, temperature, and streaming — the three things that explain most
   surprising LLM behaviour.

**Before you start:** copy `.env.example` to `.env` and paste in your OpenRouter key
(free, no card: <https://openrouter.ai> → Keys).

In [ ]:
# One-time install, if you have not already run pip install -r requirements.txt
# !pip install -q -r ../requirements.txt

In [ ]:
from shared import preflight, ask, chat, stream, client, MODEL

preflight()   # tells you exactly what this machine can reach

### Which free models exist today?

The free tier rotates: models appear and are retired without notice, and a lab that worked
last month can fail with a 404 on the model id. This cell asks OpenRouter what is currently
free **and** supports tool calling — Labs 3, 5 and 6 need that second property.

If your `.env` model is not in the list, pick one from it and edit `.env`.

In [ ]:
import requests

try:
    models = requests.get("https://openrouter.ai/api/v1/models", timeout=30).json()["data"]
    free_with_tools = [m for m in models
                       if m["id"].endswith(":free")
                       and "tools" in (m.get("supported_parameters") or [])]
    print(f"{len(free_with_tools)} free models with tool support:\n")
    for m in sorted(free_with_tools, key=lambda m: m["id"]):
        print(f"  {m['id']:<48} context {m.get('context_length', 0):>9,}")
    print(f"\nYour .env is set to: {MODEL}")
    print("STILL AVAILABLE" if any(m["id"] == MODEL for m in models)
          else "NOT IN THE LIST -> edit .env and pick one above")
except Exception as exc:
    print(f"could not reach the model list ({type(exc).__name__}) — "
          "check https://openrouter.ai/models?q=free in a browser instead")

## 1. The API is smaller than you think

A chat call is: a list of messages, a model name, and a temperature. That is the whole
interface. `shared.py` wraps it in ~10 lines — open the file, it is worth two minutes.

The **system** message sets persona, scope and rules. The **user** message carries the
request. Everything you will build in the next six labs is a way of deciding what goes
into those two strings.

In [ ]:
# TODO: ask the model to explain what an agent is, as a system prompt + user prompt.
# Use `ask(prompt, system=...)`. Give it a persona and a hard constraint (e.g. two sentences,
# no bullet points, aimed at a sceptical backend engineer).

system = ""   # TODO
prompt = ""   # TODO

print(ask(prompt, system=system))

### Same call, your own laptop

If you installed Ollama (`ollama pull llama3.2`), the identical code runs locally — same
API, different base URL. Compare the answers: the local 3B model is meaningfully worse, and
that gap is the entire argument for hosted frontier models.

In [ ]:
# TODO: run the same prompt locally with local=True. Skip this cell if you did not install Ollama.

## 2. Tokens

Tokens are the unit of price, of latency, and of the context limit. They are also why the
model cannot count the letters in "strawberry" — it never sees letters.

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")   # close enough for intuition across models

samples = {
    "english":  "The deployment failed because the database connection pool was exhausted.",
    "json":     '{"status": "failed", "reason": "connection_pool_exhausted", "retries": 3}',
    "code":     "def retry(fn, n=3):\n    for i in range(n):\n        try:\n            return fn()\n        except Exception:\n            continue",
    "russian":  "Развёртывание не удалось: пул соединений с базой данных исчерпан.",
}
for name, text in samples.items():
    n = len(enc.encode(text))
    print(f"{name:8} {len(text):4d} chars  {n:4d} tokens  {len(text)/n:.1f} chars/token")

In [ ]:
# TODO: paste in ~200 words of your own team's text — a runbook page, a ticket, a README —
# and print the token count. Then compute what 10,000 calls a day would cost at $3 per
# million input tokens. That number is usually the moment an architecture decision changes.

my_text = """
"""
# TODO: token count and daily cost

## 3. Temperature

Temperature scales how much the sampler respects the model's probabilities. Near 0 it takes
the most likely token every time; at 1.0 it samples broadly. Run the next cell and watch
the difference — this is why you cannot write an exact-match unit test.

In [ ]:
# TODO: ask the same question 3 times at temperature=0 and 3 times at temperature=1.0,
# and print the answers. Suggested prompt: "Name one risk of putting an LLM in a support inbox."

# TODO

**Note what you saw.** Temperature 0 is *more repeatable*, not reproducible: the same
prompt can still drift across runs, batches, and model updates. Testing strategy in Lab 2.

## 4. Streaming

Generation is sequential, so progress is free to display. Any user-facing feature should
stream; anything batch should not bother.

In [ ]:
# TODO: use `stream(...)` to print a longer answer as it arrives.
# Hint: for chunk in stream(prompt): print(chunk, end="", flush=True)

# TODO

## Stretch goals

1. Swap `MODEL` in `.env` for a different free model (<https://openrouter.ai/models?q=free>)
   and re-run section 1. Note which ones ignore your formatting constraint.
2. Time a call with 50 input tokens vs 5,000 input tokens, same output length. Which budget
   dominates latency?
3. Ask a question you know the answer to, about something obscure in your own domain.
   Judge the answer. That is your first, informal eval — Lab 2 makes it a real one.